In [1]:
import jax
jax.clear_caches()
print(jax.devices())

[CudaDevice(id=0), CudaDevice(id=1), CudaDevice(id=2), CudaDevice(id=3)]


In [2]:
from typing import Tuple, List
import math
from operator import add
import numpy as onp  
import jax.numpy as jnp
from jax import grad, jit, value_and_grad, random, lax, device_get
from functools import partial
from dataclasses import dataclass
from jax.tree_util import register_dataclass
from jax.tree_util import tree_map
from typing import Any



@register_dataclass
@dataclass
class AttnParams:
    w:     jnp.ndarray        # your attn_w   (d,)

@register_dataclass
@dataclass
class FfnParams:
    in_W: jnp.ndarray       # (m, d)
    out_W: jnp.ndarray      # (d, m)

@register_dataclass
@dataclass
class EmbeddingParams:
    x:    jnp.ndarray          # (V, d)
    y:    jnp.ndarray          # (V, d)
    trig: jnp.ndarray       # (d, )


@register_dataclass
@dataclass
class ModelParams:
    attn: AttnParams
    ffn:  FfnParams



def activation_function(x):
    return  0.7 * (x ** 2 - 1)/jnp.sqrt(2) + 0.3 * (x ** 3 - 3 * x)/jnp.sqrt(6)

def make_perm_params(key, V: int):
    # 1) sample the shift b uniformly
    key, k1 = random.split(key)
    b = int(random.randint(k1, (), 0, V))

    # 2) sample a until gcd(a, V) == 1
    while True:
        key, k2 = random.split(key)
        a = int(random.randint(k2, (), 1, V))
        if math.gcd(a, V) == 1:
            break

    return   a, b

def generate_data(key, V=50, N=300, L=1):
    key, *subkeys = jax.random.split(key, 6)
    perm_a, perm_b = make_perm_params(subkeys[0], V)

    latent_train = jax.random.choice(subkeys[1], V, shape=(N,))
    targets = (perm_a * latent_train + perm_b) % V 

    latent_test = jnp.arange(V)
    test_targets = (perm_a * latent_test + perm_b) % V 

    if L == 1:
        inputs = latent_train[:, None]
        test_inputs = latent_test[:, None]
    else:
        noise_tokens = random.choice(subkeys[3], V, shape=(N, L - 1))
        inputs = jnp.concatenate((latent_train[:, None], noise_tokens), axis=1)

        noise_tokens = random.choice(subkeys[4], V, shape=(V, L - 1))
        test_inputs = jnp.concatenate((latent_test[:, None], noise_tokens), axis=1)

    return (perm_a, perm_b), inputs, targets, test_inputs, test_targets, key


In [3]:
def generate_data(key, V=50, N=300, L=1):
    key, *subkeys = jax.random.split(key, 6)
    perm_a, perm_b = make_perm_params(subkeys[0], V)

    latent_train = jax.random.choice(subkeys[1], V, shape=(N,))
    targets = (perm_a * latent_train + perm_b) % V 

    latent_test = jnp.arange(V)
    test_targets = (perm_a * latent_test + perm_b) % V 

    if L == 1:
        inputs = latent_train[:, None]
        test_inputs = latent_test[:, None]
    else:
        noise_tokens = random.choice(subkeys[3], V, shape=(N, L - 1))
        inputs = jnp.concatenate((latent_train[:, None], noise_tokens), axis=1)

        noise_tokens = random.choice(subkeys[4], V, shape=(V, L - 1))
        test_inputs = jnp.concatenate((latent_test[:, None], noise_tokens), axis=1)

    return (perm_a, perm_b), inputs, targets, test_inputs, test_targets, key


def generate_embedding(key, V=50, d=100, snr = 0.1):
    key, *subkeys = jax.random.split(key,4)
    embedding_x = (1 / jnp.sqrt(d)) * jax.random.normal(subkeys[0], (V, d))
 
     
    trigger = (snr / jnp.sqrt(d)) * jax.random.normal(subkeys[1], (d, ))
    
    embedding_y = (1 / jnp.sqrt(d)) * jax.random.normal(subkeys[2], (V, d))
    return embedding_x, trigger, embedding_y, key

def attention_layer(params_attn, emb, trigger):
    emb_trig         = emb.at[:, 0, :].add(trigger)
    pre_activation   = jnp.einsum('ntd,d->nt', emb_trig, params_attn.w)   # (N, L, d) -> (N,L)
    pre_activation_ln = pre_activation #layer_norm( pre_activation, params_attn.bn,  eps=1e-5)
    attention_scores = jax.nn.softmax(pre_activation_ln , axis=1) 
    o = jnp.einsum('nt,ntd->nd', attention_scores, emb)   # (N,d)
    return  o


def linear_layer(params_ffn, x_repr):
    return  x_repr @  params_ffn.in_W.T     # (N, d)



def non_linear_layer(params_ffn, x_repr):  
    ## TODO: Complete here
    k = 3 
    h = (1/jnp.sqrt(k)) * activation_function(x_repr @  params_ffn.in_W.T)    # (N, m)
    o =  h @ params_ffn.out_W.T  # (N, d)
    return  o
 
def logistic_loss_expired(h_ln, emb_y, labels):
    true_logits = jnp.sum(h_ln * emb_y[labels], axis=-1) 
    logZ        = _logsumexp_chunked(h_ln, emb_y)  
    return jnp.mean(-true_logits + logZ)  

def logistic_loss(probs,  labels):
    batch_size = probs.shape[0]
    correct_probs = probs[jnp.arange(batch_size), labels]
    cross_entropy = -jnp.mean(jnp.log(correct_probs + 1e-12))
    return cross_entropy 


In [4]:
def compute_acc(
    params:      ModelParams,
    emb_x:       jnp.ndarray,
    emb_y:       jnp.ndarray,
    trig:        jnp.ndarray,
    labels:      jnp.ndarray
) -> Tuple[jnp.ndarray, jnp.ndarray]:
    
     # unpack
    attn = params.attn
    ffn  = params.ffn

 
    features    = attention_layer(attn, emb_x, trig)
    
     
    h    = non_linear_layer(ffn,  features)
    h_ln = h # layer_norm( h, ffn.bn,  eps=1e-5)
    
    logits = h_ln @  emb_y 
    probs  = jax.nn.softmax(logits)      
    preds = jnp.argmax(probs, axis=1)
    acc   = jnp.mean(preds == labels) 
 
    return  acc 



def compute_loss(
    params:      ModelParams,
    emb_x:       jnp.ndarray,
    emb_y:       jnp.ndarray,
    trig:        jnp.ndarray,
    labels:      jnp.ndarray,
    lam:         float = 0.0 
) -> Tuple[jnp.ndarray, jnp.ndarray]:
    
     # unpack
    attn = params.attn
    ffn  = params.ffn

 
    features    = attention_layer(attn, emb_x, trig)
    
     
    h    = non_linear_layer(ffn,  features)
    h_ln = h # layer_norm( h, ffn.bn,  eps=1e-5)
    
    logits = h_ln @  emb_y 
    probs  = jax.nn.softmax(logits)   
    loss   = logistic_loss(probs, labels)
    reg_term   = 0.5 * lam * jnp.sum(ffn.out_W ** 2)
    return loss + reg_term 


_grad_params   = jit(value_and_grad(compute_loss))



def make_multi_device_step(
    embeds:         EmbeddingParams,
    micro_batch:    int,  
    lam:            float = 0.0 
):
 
    devices = jax.local_devices()
    D = len(devices)

    emb_x = embeds.x
    emb_y = embeds.y
    trig  = embeds.trig
    
    @partial(jax.pmap,
             axis_name="devices",
             in_axes=(0,    # key per-device
                      None, # params broadcast
                      0,    # ids_shard
                      0))  
    def _step(
        key:          jnp.ndarray,
        params:       ModelParams,  
        ids_shard:    jnp.ndarray,
        labels_shard: jnp.ndarray 
    ):

        # _, g_local   = _grad_params(params,  emb_x[ids_shard], emb_y.T, trig, labels_shard, lam)
        N  = ids_shard.shape[0] 

        B =  micro_batch
        P     = (N + B - 1) // B
        total = B * P
        idx     = jnp.arange(total) % N 
 
        batches = idx.reshape((B, P))
        # compute each device’s local chunked gradient

        g0 = tree_map( jnp.zeros_like, params)

        def body(g_acc, batch_idx):
            
            train_ind_b = ids_shard[batch_idx]
            train_lbls_b= labels_shard[batch_idx]
            _, g   = _grad_params(params,  emb_x[train_ind_b], emb_y.T, trig, train_lbls_b, lam)

            gres =  tree_map(add, g_acc, g)
            return  gres, None

        g_total, _ = lax.scan(body, g0, batches)
        g_local    = tree_map(lambda x: x / B, g_total)

        # # synchronous all‐reduce (mean) across devices
        g_full = lax.pmean(g_local, axis_name="devices")

        # advance RNG
        new_key = random.split(key, 1)[0]
        return g_full, new_key

    return _step

 
def grad_compute(
    key, params, ids, labels, multi_step=None,
):
    """
    Compute the gradient of the loss w.r.t. params using multi-GPU data parallelism.

    multi_step: the pmapped function returned by make_multi_device_step,
                built once per run (see run_one below).
    """


    devices = jax.local_devices()
    D = len(devices)

    # ids: (N, T), labels: (N,)
    N, L = ids.shape
    # N_pad = ((N + D - 1) // D) * D

    # if N_pad > N:
    #     pad = N_pad - N
    #     idx = jnp.arange(pad) % N   # wrap around safely even when N < D
    #     ids    = jnp.concatenate([ids,    ids[idx]],    axis=0)
    #     labels = jnp.concatenate([labels, labels[idx]], axis=0)
    #     N = N_pad

    # # shard across devices: (D, shard, ...)
    shard = N // D
    ids_shard    = ids.reshape((D, shard, L))
    labels_shard = labels.reshape((D, shard))

    # split RNG for each device
    keys = random.split(key, D)

    # call the pre-built pmapped step
    g_accs, new_keys = multi_step(
        keys,
        params,
        ids_shard,
        labels_shard,
    )

    # gradients are identical on all devices after pmean, take device 0
    g_res = tree_map(lambda x: x[0], g_accs)
    return g_res, new_keys[0]




def _test_metrics(embeds):      
    emb_x = embeds.x
    emb_y = embeds.y
    trig  = embeds.trig


    def test_accuracy(params,test_ids, test_labels):
        return compute_acc(params, emb_x[test_ids] , emb_y.T, trig, test_labels)
    
    return test_accuracy
 

def test_metrics(key, params,  embeds, ids, lbls,  partition_size = 128):
    N   = ids.shape[0]
    idx = jnp.arange(N)
 
    B     = int(jnp.ceil(N / partition_size))
    total = B * partition_size
    pad   = total - N         # how many extra we need
 
    if pad > 0:
        key, sub = jax.random.split(key)
        # sample with replacement
        extra = jax.random.choice(sub, N, shape=(pad,), replace=True)
        idx   = jnp.concatenate([idx, extra], axis=0)

    test_accuracy = _test_metrics(embeds)
    batches = idx.reshape((B, partition_size))

    def test_body(carry, batch_ids):
            acc = carry
            test_ids_b =  ids[batch_ids]
            test_lbl_b =  lbls[batch_ids]
        
            at = test_accuracy(params, test_ids_b, test_lbl_b)
            acc += at

            return acc, None

    acc, _ = lax.scan(test_body, 0.0, batches)

    return  acc/B, key


In [ ]:
if __name__ == "__main__":
    
    def run_one(key, V: int = 50, N: int = 250, d: int = 40, L: int = 1, num_epochs: int = 1500, lr_alg: float = 0.1, iter_per_epoch: int = 2) -> float:

        # Initialize parameters
        k = 3
        m = int(  (d ** k)/4 )
        # print(m)
        key, k1 = random.split(key)
        

        
        params = ModelParams(
                    attn=AttnParams(
                        w= jnp.zeros(d) ),
                    ffn=FfnParams(
                        in_W  = jax.random.normal(k1, shape = (m,d) ),
                        out_W = jnp.zeros((d, m)  ) )
                )


        
        emb_x, trigger, emb_y, key = generate_embedding(key, V, d, snr = 0.4) # 0.1 if small signal, 0.2 nothing, 0.5 strong signal  
        embeds = EmbeddingParams( x =  emb_x, y = emb_y, trig = trigger)
        perm, train_ids, train_lbls, test_ids, test_lbls, key = generate_data(key, V, N, L)

        mb = 1
        partition_size = 128
        mb = 2048
        if jnp.log2(V) > 10.1 and jnp.log2(d) > 7.1:
            mb = 4096
            partition_size = 128

        multi_step = make_multi_device_step(
            embeds=embeds,
            micro_batch = mb,
            lam= 5e-15,
        )
 
        
        
        # Hyperparameters
        sqrt_V_over_m =  jnp.sqrt(jnp.asarray(V, float) / jnp.asarray(m, float))
        attn_lr =  2e8 * (L ** 2) *  sqrt_V_over_m
        lr = lr_alg
        batch_size  = N

        acc_res = jnp.zeros(4)
        acc_res = acc_res.at[0].set(0)
        idx = 1
        
        # First step
        gp, key = grad_compute(
                key,
                params,
                ids=train_ids,
                labels=train_lbls,
                multi_step=multi_step,    
            )
        
        params.ffn.out_W =  params.ffn.out_W - lr * gp.ffn.out_W
        # print("Out W1", jnp.mean(params.ffn.out_W[:] ** 2))

        acc, key = test_metrics(key, params,  embeds, test_ids, test_lbls, partition_size)
        acc_res = acc_res.at[idx].set(acc)
        idx = idx + 1
        
        # Second step
        gp, key = grad_compute(
                key,
                params,
                ids=train_ids,
                labels=train_lbls,
                multi_step=multi_step,    
            )

        params.attn.w =  params.attn.w - attn_lr * gp.attn.w
 

        acc, key = test_metrics(key, params,  embeds, test_ids, test_lbls, partition_size)
        acc_res = acc_res.at[idx].set(acc)
        idx = idx + 1

        # Third step
        gp, key = grad_compute(
                key,
                params,
                ids=train_ids,
                labels=train_lbls,
                multi_step=multi_step,    
            )
        
        params.ffn.out_W =  params.ffn.out_W - 100 * sqrt_V_over_m * gp.ffn.out_W

        acc, key = test_metrics(key, params,  embeds, test_ids, test_lbls, partition_size)
        acc_res = acc_res.at[idx].set(acc)
        idx = idx + 1
  
        return   params, acc_res,  key 



    
    seed     = 14
    key      = jax.random.PRNGKey(seed)
    devices  = jax.local_devices()
    D        = len(devices)
    iter_per_epoch = 1
    print("Device size:", D, "Iteration per epoch:", iter_per_epoch)
 
      
    Vlis      = (2.0 **  jnp.linspace(13.0,16.0, num= 30) ).astype(int)  
    dlis      = jnp.ceil(2.0 ** jnp.linspace(6.0,8.0, num= 20)).astype(int) 
    Llis      = [None]
    Nlis      = [None]
    repeat    = 3
    lr        = 0.0005
    num_epoch = 1

    num_runs  = len(Vlis) * len(Nlis) *  len(Llis) *   len(dlis) * repeat
    
    key, *subkeys  = random.split(key, num_runs + 1)
    idx      = 0
 
    # eff_num_runs  = len(Vlis) * len(Nlis) *  len(Llis) *   len(dlis) * repeat
    # subkeys       = subkeys[-eff_num_runs:]
    # num_runs      = eff_num_runs

    Deff =  (D* iter_per_epoch)
    Ns   =   Vlis *  (jnp.log(Vlis)  + 5.1 )
    Ns   =  (  jnp.round(  Ns   /  Deff )  * Deff   ).astype(int)
    Ls   =  ( (Vlis ** 0.5)  ).astype(int)
    print("Vlis", Vlis)
    print("Nlis", Ns)
    print("Llis", Ls)

    
    shape        = (len(Vlis), len(Nlis), len(Llis), len(dlis), repeat, num_epoch * 3 + 1)
    accuracy_vec = jnp.zeros(shape, dtype=jnp.float32)
    repeat_vec   =  jnp.zeros(repeat, dtype=jnp.float32)
 

    signal_vec =  jnp.zeros(( len(dlis),repeat), dtype=jnp.float32)
    noise_vec  =  jnp.zeros(( len(dlis),repeat), dtype=jnp.float32)
    
    
    
    for i, Vcurrent in enumerate(Vlis):
        for j, Nval in enumerate(Nlis):
            Ncurrent =  Vcurrent *  (jnp.log(Vcurrent)  + 5.1 )
            # Ncurrent =  ( jnp.round(  Ncurrent   /   batch_size ) * batch_size   ).astype(int)
            Ncurrent =  (  jnp.round(  Ncurrent  /  Deff )  * Deff   ).astype(int)
            for l, Lval in enumerate(Llis):
                if Lval is None:
                    Lcurrent =  int(   jnp.sqrt(Vcurrent)) # int( (Vcurrent ** 0.75)/8 )  
  
                for k, dcurrent in enumerate(dlis):
                    for r in range(repeat):
                    
                        subkey = subkeys[idx]
                        param_final, acc_res, _  = run_one(
                            subkey, Vcurrent, Ncurrent,
                            dcurrent, Lcurrent,
                            num_epoch, lr, iter_per_epoch
                        )
                        repeat_vec = repeat_vec.at[r].set(float(acc_res[-1]))

                        # print("Attn W", jnp.mean(param_final.attn.w[:] ** 2))
                        # print("Out W2", jnp.mean(param_final.ffn.out_W[:] ** 2))
                        

                        print(f"[{idx+1}/{num_runs}] V={Vcurrent}, N={Ncurrent}, "
                          f"L={Lcurrent}, d={dcurrent}, r={r}", "Accuracy", acc_res[1:])

                        idx += 1
                        jax.clear_caches()
                        
                        accuracy_vec = accuracy_vec.at[i, j, l, k, r, :].set(acc_res)


                    if  jnp.median(repeat_vec) > 0.3:
                        accuracy_vec = accuracy_vec.at[i, j, l, k:, :, :].set(acc_res )
                        print("Skipping d =",dlis[k:])
                        idx += len(dlis[k:]) * repeat - repeat
                        break


  
                    
                    host_acc =  device_get(accuracy_vec)
                    onp.savez_compressed(
                        "checkpoint_nonlinear_accuracy_vec_gd_LV05_md3_s04.npz", 
                        accuracy_vec=host_acc,
                        Vlis=Vlis, Nlis=Ns, dlis=dlis, Llis=Ls,
                        seed=seed, lr=lr,
                        device_size = D,
                        part_size = iter_per_epoch
                        # batch_size = batch_size
                    )

    print("Done")

  


Device size: 4 Iteration per epoch: 1


/mnt/sw/nix/store/71ksmx7k6xy3v9ksfkv5mp5kxxp64pd6-python-3.10.13-view/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/mnt/sw/nix/store/71ksmx7k6xy3v9ksfkv5mp5kxxp64pd6-python-3.10.13-view/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


Vlis [ 8192  8800  9455 10158 10913 11724 12596 13532 14538 15619 16780 18027
 19367 20807 22354 24016 25801 27719 29780 31994 34372 36927 39672 42622
 45790 49194 52851 56780 61001 65536]
Nlis [ 115596  124808  134776  145524  157124  169640  183160  197740  213484
  230476  248812  268596  289948  313000  337876  364720  393676  424928
  458660  495052  534312  576676  622392  671728  724940  782360  844308
  911144  983252 1061052]
Llis [ 90  93  97 100 104 108 112 116 120 124 129 134 139 144 149 154 160 166
 172 178 185 192 199 206 213 221 229 238 246 256]


2026-02-25 02:51:12.373141: E external/xla/xla/service/rendezvous.cc:92] [id=2] This thread has been waiting for `initialize clique for rank 0; clique=devices=[0,1,2,3]; stream=0; groups=[[0,1,2,3]]; root_device=-1; num_local_participants=4; run_id=-206919558` for 10 seconds and may be stuck. All 4 threads joined the rendezvous, however the leader has not marked the rendezvous as completed. Leader can be deadlocked inside the rendezvous callback.
2026-02-25 02:51:12.373224: E external/xla/xla/service/rendezvous.cc:92] [id=1] This thread has been waiting for `initialize clique for rank 1; clique=devices=[0,1,2,3]; stream=0; groups=[[0,1,2,3]]; root_device=-1; num_local_participants=4; run_id=-206919558` for 10 seconds and may be stuck. All 4 threads joined the rendezvous, however the leader has not marked the rendezvous as completed. Leader can be deadlocked inside the rendezvous callback.
2026-02-25 02:51:12.373241: E external/xla/xla/service/rendezvous.cc:92] [id=0] This thread has be

[1/1800] V=8192, N=115596, L=90, d=64, r=0 Accuracy [1.22070312e-04 2.44140625e-04 1.23046875e-01]
[2/1800] V=8192, N=115596, L=90, d=64, r=1 Accuracy [0.00012207 0.00012207 0.09301758]
[3/1800] V=8192, N=115596, L=90, d=64, r=2 Accuracy [0.00012207 0.00012207 0.05895996]
[4/1800] V=8192, N=115596, L=90, d=69, r=0 Accuracy [1.2207031e-04 0.0000000e+00 1.8640137e-01]
[5/1800] V=8192, N=115596, L=90, d=69, r=1 Accuracy [1.2207031e-04 0.0000000e+00 1.4050293e-01]
[6/1800] V=8192, N=115596, L=90, d=69, r=2 Accuracy [1.2207031e-04 0.0000000e+00 2.6159668e-01]
[7/1800] V=8192, N=115596, L=90, d=75, r=0 Accuracy [1.2207031e-04 0.0000000e+00 2.8881836e-01]
[8/1800] V=8192, N=115596, L=90, d=75, r=1 Accuracy [1.2207031e-04 2.4414062e-04 1.5954590e-01]
[9/1800] V=8192, N=115596, L=90, d=75, r=2 Accuracy [1.2207031e-04 0.0000000e+00 3.4082031e-01]
[10/1800] V=8192, N=115596, L=90, d=80, r=0 Accuracy [1.2207031e-04 2.4414062e-04 3.4362793e-01]
[11/1800] V=8192, N=115596, L=90, d=80, r=1 Accuracy [